# Introduction to Table Clustering in Snowflake

In [ ]:
DROP DATABASE IF EXISTS DEMO_DB;

## Step 1: Setup

In [ ]:
-- Create new database and schema

CREATE DATABASE IF NOT EXISTS DEMO_DB;
CREATE SCHEMA IF NOT EXISTS DEMO_SCHEMA;
USE DATABASE DEMO_DB;
USE SCHEMA DEMO_SCHEMA;
CREATE WAREHOUSE IF NOT EXISTS DEMO_WH WITH WAREHOUSE_SIZE = 'XSMALL' AUTO_SUSPEND = 300;
USE WAREHOUSE DEMO_WH;

## Step 2: Create Non-Clustered Table

In [ ]:
-- Non-clustered table
CREATE OR REPLACE TRANSIENT TABLE sales_non_clustered (
    sale_id INT,
    product_name STRING,
    amount DECIMAL(10,2),
    sale_date DATE
);

In [ ]:
/*
Objective: Populate tables with sample data.
Instructions:
Insert 50,000,000 rows into both tables, ensuring sale_id and amount values fit within their respective data types (INT and DECIMAL(10,2)):
*/

INSERT INTO sales_non_clustered
    SELECT 
        (SEQ4() % 2147483647) + 1 AS sale_id, -- Fits INT range
        CASE WHEN RANDOM() < 0.5 THEN 'Laptop' ELSE 'Phone' END AS product_name,
        UNIFORM(0, 9999, RANDOM()) / 100.0 AS amount, -- Fits DECIMAL(10,2), e.g., 0.00 to 9999.99
        DATEADD('DAY', -(RANDOM() % 365), '2025-09-21') AS sale_date
    FROM TABLE(GENERATOR(ROWCOUNT => 50000000)); --50 mln rows

## Step 3: Create Clustered Table (as select from non-clustered)

In [ ]:
-- Clustered table
CREATE OR REPLACE TRANSIENT TABLE sales_clustered
CLUSTER BY (sale_date)
AS SELECT * FROM sales_non_clustered


## Step 4: Query Performance Comparison

In [ ]:
-- Non-clustered

SELECT product_name, SUM(amount) AS total_sales
FROM sales_non_clustered
WHERE sale_date BETWEEN '2025-01-01' AND '2025-01-02'
GROUP BY product_name;

In [ ]:
-- Clustered

SELECT product_name, SUM(amount) AS total_sales
FROM sales_clustered
WHERE sale_date BETWEEN '2025-01-01' AND '2025-01-02'
GROUP BY product_name;

In [ ]:
EXPLAIN
SELECT product_name, SUM(amount) AS total_sales
FROM sales_clustered
WHERE sale_date BETWEEN '2025-01-01' AND '2025-01-02'
GROUP BY product_name
->>
SELECT "operation", "partitionsTotal", "partitionsAssigned", "bytesAssigned"
FROM $1
WHERE "operation" = 'TableScan';

In [ ]:
EXPLAIN
SELECT product_name, SUM(amount) AS total_sales
FROM sales_non_clustered
WHERE sale_date BETWEEN '2025-01-01' AND '2025-01-02'
GROUP BY product_name
->>
SELECT "operation", "partitionsTotal", "partitionsAssigned", "bytesAssigned"
FROM $1
WHERE "operation" = 'TableScan';

In [ ]:
WITH x AS (
  SELECT 
    PARSE_JSON(SYSTEM$CLUSTERING_INFORMATION('DEMO_DB.DEMO_SCHEMA.SALES_CLUSTERED', '(SALE_DATE)')) AS meta
)
SELECT 
  meta:total_partition_count AS total_partition_count,
  meta:average_depth AS average_depth
FROM x;


In [ ]:
WITH x AS (
  SELECT 
    PARSE_JSON(SYSTEM$CLUSTERING_INFORMATION('DEMO_DB.DEMO_SCHEMA.SALES_NON_CLUSTERED', '(SALE_DATE)')) AS meta
)
SELECT 
  meta:total_partition_count AS total_partition_count,
  meta:average_depth AS average_depth
FROM x;

## Step 5: Cleanup

In [ ]:
-- DROP DATABASE IF EXISTS DEMO_DB;
-- DROP WAREHOUSE IF EXISTS DEMO_WH;
-- USE WAREHOUSE COMPUTE_WH;

In [ ]:
df =  clustering_info_clustered.to_df()

In [ ]:
df.show()